In [1]:
import config as cfg
from data_acquirer import get_data, gen_data
from data_processor import process
import helpers
import heuristic
from heuristic import Path
from solver import solve_pdpcr

import copy
import datetime
from docplex.mp.model import Model
import logging
import heapq
import numpy as np
import time

logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')

In [10]:

#WR2 data
sec_names = ['100','200','400','600','800','1000']
third_names = ['1','2','3','4','5']


data_paths =[f'wr2/bar-n{j}-{k}-3.csv'for j in sec_names for k in third_names]#TODO add data
dist_paths = [f'wr2/bar-n{j}-{k}-dist.csv'for j in sec_names for k in third_names]#TODO add data
num_robots = [5*i for i in range(2,3)]

method = 5
file_type = 'csv'
is_depot_task = False


In [2]:

#Li larger data

first_names = ['LC1']
sec_names = [2,4,6,8,10]
third_names = [i for i in range(1,11)]
data_paths =[f'lim/{j}00/{i}_{j}_{k}-5-uniform.csv'for i in first_names for j in sec_names for k in third_names]#
num_robots = [10*i for i in range(1,4)]
method = 5
file_type = 'csv'
is_depot_task = False

In [ ]:
num_robots = np.random.randint(1,[2,3,4,5,6,7,8,9,10,11],[10,10])
num_robots

In [14]:
num_robots_fixed = np.array([[ 1,  2,  3,  1,  1,  2,  3,  3,  2,  9],
       [ 1,  1,  1,  1,  1,  1,  3,  4,  8,  8],
       [ 1,  2,  2,  1,  2,  3,  5,  4,  4, 10],
       [ 1,  1,  3,  4,  1,  2,  7,  1,  6,  6],
       [ 1,  1,  1,  3,  5,  2,  3,  6,  7,  7],
       [ 1,  1,  3,  2,  1,  5,  5,  4,  4,  9],
       [ 1,  2,  3,  2,  4,  1,  4,  7,  7,  6],
       [ 1,  1,  3,  4,  3,  6,  3,  2,  8,  4],
       [ 1,  2,  1,  1,  5,  3,  3,  4,  4,  9],
       [ 1,  2,  2,  3,  4,  4,  7,  5,  9,  7]])

num_robots_fixed2 = np.array([[1, 1, 2, 1, 5, 1, 7, 7, 4, 3],
       [1, 2, 1, 3, 5, 3, 3, 2, 4, 4],
       [1, 1, 2, 1, 4, 1, 4, 4, 1, 7],
       [1, 2, 3, 3, 2, 5, 5, 8, 9, 8],
       [1, 1, 3, 4, 4, 6, 6, 3, 6, 4],
       [1, 1, 1, 4, 1, 2, 5, 2, 5, 5],
       [1, 1, 2, 1, 5, 6, 1, 6, 4, 1],
       [1, 1, 1, 4, 2, 2, 3, 5, 6, 5],
       [1, 2, 1, 4, 4, 2, 2, 4, 4, 6],
       [1, 2, 1, 3, 1, 2, 5, 8, 9, 4]])

In [ ]:
# CPLEX data generated
data_idcs = [4,5,16,18]
data_paths = [f'generated{i}.csv' for i in data_idcs]
num_robots = [10]
file_type = 'csv'
i = 0
cfg.SOLVE_TIME =900
is_depot_task = False
for data_path in data_paths:
    for num_robot in num_robots:
        for i in range(1):
            try:
                # cur_num_robots = []
                # cur_num_robots = num_robots_fixed2[:,i]
        

                raw = get_data(data_path, file_type, is_depot_task)
                avg_num_robots = np.mean([raw.tasks[i].n for i in range(len(raw.tasks))])
                print(f'Solving for {data_path} with {num_robot} robots with average n_j = {avg_num_robots}')
                method = 5
                processed_data_vrp = process(raw, method)
                solve_result_vrp = solve_pdpcr(processed_data_vrp, num_robot, method)
                # method = 2
                # processed_data_flow_old = process(raw, method)
                # solve_result_flow_old = solve_pdpcr(processed_data_flow_old, num_robot, method)
                

                solve_log = []
                solve_log.append(data_path)
                solve_log.append(num_robot)

                # solve_log.append(solve_result_flow_old['solve_time'])
                # solve_log.append(solve_result_flow_old['sol'].objective_value)
                # solve_log.append(solve_result_flow_old['processed_nodes'])
                # solve_log.append(solve_result_flow_old['gap'])

                solve_log.append(solve_result_vrp['solve_time'])
                solve_log.append(solve_result_vrp['sol'].objective_value)
                solve_log.append(solve_result_vrp['processed_nodes'])
                solve_log.append(solve_result_vrp['gap'])

                now = datetime.datetime.now()
                solve_log.append(avg_num_robots)
                solve_log.append(now.strftime("%Y-%m-%d %H:%M:%S"))
                with open('solve_results/solve_log_li_lim_vrp.csv', 'a') as f:
                    f.write(','.join([str(i) for i in solve_log])+'\n')
            except Exception as e:
                print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
                print(e)
            i += 1

In [ ]:
# read file 'solve_results/solve_log_generated_progressive_large_nj.csv' to pandas dataframe, it has the following columns:
# data_name, total_robots, solve_time_vrp, obj_value_vrp, processed_nodes_vrp, gap_vrp, solve_time_flow, obj_value_flow, processed_nodes_flow, gap_flow, avg_num_robots,date
import pandas as pd
df_large_nj = pd.read_csv('solve_results/solve_log_generated_progressive_large_nj.csv', header=None)
df_large_nj.columns = ['data_name', 'total_robots', 'solve_time_vrp', 'obj_value_vrp', 'processed_nodes_vrp', 'gap_vrp', 'solve_time_flow', 'obj_value_flow', 'processed_nodes_flow', 'gap_flow','avg_num_robots', 'date']


In [3]:
first_names = ['lc','lr','lrc']
sec_names = [i for i in range(101,115)] +[i for i in range(201,215)]
# third_names = [i for i in range(1,11)]
data_paths =[f'lim/100/{i}{j}-5-uniform.csv'for i in first_names for j in sec_names]#
num_robots = [10*i for i in range(1,2)]
file_type = 'csv'
is_depot_task = False

In [ ]:
## Mass solve using tabu_search
# data_idcs = [16,18,19]
# data_paths = [f'generated{i}.csv' for i in data_idcs]
num_robots = [10]
file_type = 'csv'
is_depot_task = False
note = 'Tabu search with 2 original neighborhoods'

for num_robot in num_robots:
    for data_path in data_paths:
        try:
            raw = get_data(data_path, file_type, is_depot_task)
            logging.info(f'Solving data {data_path} with {num_robot} robots and {len(raw.tasks)} tasks')
            # processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            depot = processed_h.depot
            tasks = processed_h.tasks
            cfg.TABU_MAX_TL_ADD_SIZE = int(len(tasks)/5)
            cfg.TABU_MAX_TL_DROP_SIZE = int(len(tasks)/5)
            cfg.TABU_NUM_REROUTE_NEIGHBORS =   int(len(tasks)/4)
            cfg.TABU_NUM_REVERSE_NEIGHBORS =  int(len(tasks)/14)
            cfg.TABU_NUM_REDISTRIBUTE_NEIGHBORS = 0 #int(len(tasks)/15)
            # cfg.TABU_MAX_TL_ADD_SIZE = 12
            # cfg.TABU_MAX_TL_DROP_SIZE = 12
            # cfg.TABU_NUM_REROUTE_NEIGHBORS = 20
            # cfg.TABU_NUM_REVERSE_NEIGHBORS = 10
            cfg.TABU_MAX_ITER = 5000
            cfg.TABU_FIND_FROM_ELITE_AFTER = 200 #iterations
            cfg.TABU_RESTART_AFTER = 400 #iterations
            cfg.TABU_REVERSE_ON_CRITICAL = True
            cfg.TABU_REROUTE_ON_CRITICAL = True
            cfg.TABU_REROUTE_LONGEST_EDGE = True
            cfg.TABU_REDISTRIBUTE_PROB = 0.3
            cfg.TABU_SAVE_FITNESSES = True
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)

            data_name = data_path.split('/')[2].split('.')[0]
            init_solution_path = f'solve_results/init_data/{data_name}' #For fetch initial solution
            tik1 = time.time()
            tabu = heuristic.TabuSearch(
                processed_h, 
                c_travel, 
                c_operate, 
                heuristic.SolutionFetcher, 
                init_solution_path = init_solution_path)
            best_sol, best_sol_obj = tabu.search(num_robot,cfg.TABU_MAX_ITER,cfg.TABU_MAX_TL_ADD_SIZE)
            tok1 = time.time()
            tabu_time = tok1 - tik1
            print(f'Tabu time: {tabu_time}')
            if cfg.TABU_SAVE_FITNESSES:
                data_name = data_path.split('/')[2].split('.')[0]
                #add current time to data_name
                data_name += '_'+ datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
                tabu.save_fitness_progress(data_name)
        
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(best_sol_obj)
            solve_log.append(np.round(tabu_time,2))
            now = datetime.datetime.now()
            solve_log.append(now)
            solve_log.append(note)
            with open('solve_log_tabu.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

08-Oct-24 10:42:30:INFO:Solving data lim/200/LC1_2_1-5-uniform.csv with 10 robots and 106 tasks
08-Oct-24 10:42:30:INFO:TABU SEARCH PARAMETERS
08-Oct-24 10:42:30:INFO:RANDOM_SEED: 16
08-Oct-24 10:42:30:INFO:NUM_REVERSE_NEIGHBORS: 7
08-Oct-24 10:42:30:INFO:NUM_REROUTE_NEIGHBORS: 26
08-Oct-24 10:42:30:INFO:NUM_REDISTRIBUTE_NEIGHBORS: 0
08-Oct-24 10:42:30:INFO:MAX_TL_ADD_SIZE: 13
08-Oct-24 10:42:30:INFO:MAX_TL_DROP_SIZE: 13
08-Oct-24 10:42:30:INFO:MAX_TL_REDISTRIBUTE_SIZE: 10
08-Oct-24 10:42:30:INFO:MAX_ITER: 5000
08-Oct-24 10:42:30:INFO:FIND_FROM_ELITE_AFTER: 200
08-Oct-24 10:42:30:INFO:RESTART_AFTER: 400
08-Oct-24 10:42:30:INFO:REROUTE_ON_CRITICAL: True
08-Oct-24 10:42:30:INFO:REVERSE_ON_CRITICAL: True
08-Oct-24 10:42:30:INFO:REVERSE_CRITICAL_LONGEST_EDGE: True
08-Oct-24 10:42:30:INFO:Initial makespan: 937.2506795011043
08-Oct-24 10:42:30:INFO:Initial fitness: 937.2506795011043
08-Oct-24 10:42:30:INFO:Iteration 0 done
08-Oct-24 10:42:30:INFO:Best fitness updated: 935.1138551785186, chos

In [3]:
## Mass solve using tabu_search
#ONLY REDISTRIBUTE
num_robots = [10]
file_type = 'csv'
is_depot_task = False

for num_robot in num_robots:
    for data_path in data_paths:
        try:
            raw = get_data(data_path, file_type, is_depot_task)
            logging.info(f'Solving data {data_path} with {num_robot} robots and {len(raw.tasks)} tasks')
            # processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            depot = processed_h.depot
            tasks = processed_h.tasks
            cfg.TABU_MAX_TL_ADD_SIZE = int(len(tasks)/8)
            cfg.TABU_MAX_TL_DROP_SIZE = int(len(tasks)/8)
            cfg.TABU_NUM_REROUTE_NEIGHBORS =  0 #int(len(tasks)/5)
            cfg.TABU_NUM_REVERSE_NEIGHBORS = 0 # int(len(tasks)/10)
            cfg.TABU_NUM_REDISTRIBUTE_NEIGHBORS = int(len(tasks)/20)
            # cfg.TABU_MAX_TL_ADD_SIZE = 12
            # cfg.TABU_MAX_TL_DROP_SIZE = 12
            # cfg.TABU_NUM_REROUTE_NEIGHBORS = 20
            cfg.TABU_MAX_ITER = 200
            cfg.TABU_FIND_FROM_ELITE_AFTER = 200 #iterations
            cfg.TABU_RESTART_AFTER = 400 #iterations
            cfg.TABU_REVERSE_ON_CRITICAL = True
            cfg.TABU_REROUTE_ON_CRITICAL = True
            cfg.TABU_REROUTE_LONGEST_EDGE = True
            cfg.TABU_REDISTRIBUTE_PROB = 0.3
            cfg.TABU_SAVE_FITNESSES = True
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)

            data_name = data_path.split('/')[2].split('.')[0]
            init_solution_path = f'solve_results/init_data/{data_name}' #For fetch initial solution
            tik1 = time.time()
            tabu = heuristic.TabuSearch(
                processed_h, 
                c_travel, 
                c_operate, 
                heuristic.SolutionFetcher, 
                init_solution_path = init_solution_path)
            best_sol, best_sol_obj = tabu.search(num_robot,cfg.TABU_MAX_ITER)
            tok1 = time.time()
            tabu_time = tok1 - tik1
            print(f'Tabu time: {tabu_time}')
            if cfg.TABU_SAVE_FITNESSES:
                data_name = data_path.split('/')[2].split('.')[0]
                #add current time to data_name
                data_name += '_'+ datetime.datetime.now().strftime("%Y-%m-%d") + 'redis_only'
                tabu.save_fitness_progress(data_name)
        
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(best_sol_obj)
            solve_log.append(np.round(tabu_time,2))
            now = datetime.datetime.now()
            solve_log.append(now)
            with open('solve_log_tabu.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

18-Sep-24 15:27:36:INFO:Solving data lim/200/LC1_2_1-5-uniform.csv with 10 robots and 106 tasks


18-Sep-24 15:27:36:INFO:TABU SEARCH PARAMETERS
18-Sep-24 15:27:36:INFO:RANDOM_SEED: 16
18-Sep-24 15:27:36:INFO:NUM_REVERSE_NEIGHBORS: 0
18-Sep-24 15:27:36:INFO:NUM_REROUTE_NEIGHBORS: 0
18-Sep-24 15:27:36:INFO:NUM_REDISTRIBUTE_NEIGHBORS: 5
18-Sep-24 15:27:36:INFO:MAX_TL_ADD_SIZE: 13
18-Sep-24 15:27:36:INFO:MAX_TL_DROP_SIZE: 13
18-Sep-24 15:27:36:INFO:MAX_TL_REDISTRIBUTE_SIZE: 10
18-Sep-24 15:27:36:INFO:MAX_ITER: 200
18-Sep-24 15:27:36:INFO:FIND_FROM_ELITE_AFTER: 200
18-Sep-24 15:27:36:INFO:RESTART_AFTER: 400
18-Sep-24 15:27:36:INFO:REROUTE_ON_CRITICAL: True
18-Sep-24 15:27:36:INFO:REVERSE_ON_CRITICAL: True
18-Sep-24 15:27:36:INFO:REVERSE_CRITICAL_LONGEST_EDGE: True
18-Sep-24 15:27:36:INFO:Initial makespan: 937.2506795011043
18-Sep-24 15:27:36:INFO:Initial fitness: 937.2506795011043
18-Sep-24 15:27:37:INFO:Best fitness updated: 937.0341743313893, chosen neighborhood: 1
18-Sep-24 15:27:37:INFO:Iteration 0 done
18-Sep-24 15:27:38:INFO:Best fitness updated: 932.375317459882, chosen neighbor

Tabu time: 160.85776543617249


18-Sep-24 15:30:17:INFO:Best fitness updated: 1120.816358689534, chosen neighborhood: 1
18-Sep-24 15:30:17:INFO:Iteration 0 done
18-Sep-24 15:30:19:INFO:Best fitness updated: 1120.0584798678647, chosen neighborhood: 1
18-Sep-24 15:30:21:INFO:Best fitness updated: 1119.5962753505112, chosen neighborhood: 1
18-Sep-24 15:30:24:INFO:Best fitness updated: 1115.6759715156807, chosen neighborhood: 1
18-Sep-24 15:30:25:INFO:Best fitness updated: 1114.5770403148802, chosen neighborhood: 1
18-Sep-24 15:30:30:INFO:Best fitness updated: 1111.8874459922, chosen neighborhood: 1
18-Sep-24 15:30:48:INFO:Best fitness updated: 1110.484041656413, chosen neighborhood: 1
18-Sep-24 15:30:53:INFO:Iteration 50 done
18-Sep-24 15:30:54:INFO:Best fitness updated: 1109.6235317958299, chosen neighborhood: 1
18-Sep-24 15:30:58:INFO:Best fitness updated: 1108.1089278175236, chosen neighborhood: 1
18-Sep-24 15:30:59:INFO:Best fitness updated: 1107.1479554113043, chosen neighborhood: 1
18-Sep-24 15:31:01:INFO:Best fit

Tabu time: 146.2484495639801


18-Sep-24 15:32:44:INFO:Iteration 0 done
18-Sep-24 15:32:54:INFO:Best fitness updated: 1464.5738232488757, chosen neighborhood: 1
18-Sep-24 15:32:56:INFO:Best fitness updated: 1463.8276905402402, chosen neighborhood: 1
18-Sep-24 15:32:59:INFO:Best fitness updated: 1463.583266413781, chosen neighborhood: 1
18-Sep-24 15:33:20:INFO:Iteration 50 done
18-Sep-24 15:33:46:INFO:Best fitness updated: 1450.5333198313815, chosen neighborhood: 1
18-Sep-24 15:33:46:INFO:Best fitness updated: 1439.0467569127495, chosen neighborhood: 1
18-Sep-24 15:33:48:INFO:Best fitness updated: 1427.7979838761273, chosen neighborhood: 1
18-Sep-24 15:33:57:INFO:Iteration 100 done
18-Sep-24 15:34:36:INFO:Iteration 150 done
18-Sep-24 15:35:13:INFO:Best solution found: 1427.7979838761273
18-Sep-24 15:35:13:INFO:Solving data lim/200/LC1_2_4-5-uniform.csv with 10 robots and 105 tasks
18-Sep-24 15:35:14:INFO:TABU SEARCH PARAMETERS
18-Sep-24 15:35:14:INFO:RANDOM_SEED: 16
18-Sep-24 15:35:14:INFO:NUM_REVERSE_NEIGHBORS: 0
18

... (28 outputs left out)

Tabu time: 1074.4604768753052


18-Sep-24 18:05:02:INFO:TABU SEARCH PARAMETERS
18-Sep-24 18:05:02:INFO:RANDOM_SEED: 16
18-Sep-24 18:05:02:INFO:NUM_REVERSE_NEIGHBORS: 0
18-Sep-24 18:05:02:INFO:NUM_REROUTE_NEIGHBORS: 0
18-Sep-24 18:05:02:INFO:NUM_REDISTRIBUTE_NEIGHBORS: 10
18-Sep-24 18:05:02:INFO:MAX_TL_ADD_SIZE: 26
18-Sep-24 18:05:02:INFO:MAX_TL_DROP_SIZE: 26
18-Sep-24 18:05:02:INFO:MAX_TL_REDISTRIBUTE_SIZE: 10
18-Sep-24 18:05:02:INFO:MAX_ITER: 200
18-Sep-24 18:05:02:INFO:FIND_FROM_ELITE_AFTER: 200
18-Sep-24 18:05:02:INFO:RESTART_AFTER: 400
18-Sep-24 18:05:02:INFO:REROUTE_ON_CRITICAL: True
18-Sep-24 18:05:02:INFO:REVERSE_ON_CRITICAL: True
18-Sep-24 18:05:02:INFO:REVERSE_CRITICAL_LONGEST_EDGE: True
18-Sep-24 18:05:02:INFO:Initial makespan: 2011.819452383876
18-Sep-24 18:05:02:INFO:Initial fitness: 2011.819452383876
18-Sep-24 18:05:07:INFO:Iteration 0 done
18-Sep-24 18:05:43:INFO:Best fitness updated: 2011.4183310049834, chosen neighborhood: 1
18-Sep-24 18:06:07:INFO:Best fitness updated: 2010.1517378800731, chosen neig

Tabu time: 1147.337572336197


18-Sep-24 18:24:09:INFO:TABU SEARCH PARAMETERS
18-Sep-24 18:24:09:INFO:RANDOM_SEED: 16
18-Sep-24 18:24:09:INFO:NUM_REVERSE_NEIGHBORS: 0
18-Sep-24 18:24:09:INFO:NUM_REROUTE_NEIGHBORS: 0
18-Sep-24 18:24:09:INFO:NUM_REDISTRIBUTE_NEIGHBORS: 10
18-Sep-24 18:24:09:INFO:MAX_TL_ADD_SIZE: 26
18-Sep-24 18:24:09:INFO:MAX_TL_DROP_SIZE: 26
18-Sep-24 18:24:09:INFO:MAX_TL_REDISTRIBUTE_SIZE: 10
18-Sep-24 18:24:09:INFO:MAX_ITER: 200
18-Sep-24 18:24:09:INFO:FIND_FROM_ELITE_AFTER: 200
18-Sep-24 18:24:09:INFO:RESTART_AFTER: 400
18-Sep-24 18:24:09:INFO:REROUTE_ON_CRITICAL: True
18-Sep-24 18:24:09:INFO:REVERSE_ON_CRITICAL: True
18-Sep-24 18:24:09:INFO:REVERSE_CRITICAL_LONGEST_EDGE: True
18-Sep-24 18:24:09:INFO:Initial makespan: 2231.6918642865403
18-Sep-24 18:24:09:INFO:Initial fitness: 2231.6918642865403
18-Sep-24 18:24:16:INFO:Iteration 0 done


In [5]:
## Mass solve using tabu_search
#ONLY REROUTE
num_robots = [10]
file_type = 'csv'
is_depot_task = False

for num_robot in num_robots:
    for data_path in data_paths:
        try:
            raw = get_data(data_path, file_type, is_depot_task)
            logging.info(f'Solving data {data_path} with {num_robot} robots and {len(raw.tasks)} tasks')
            # processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            depot = processed_h.depot
            tasks = processed_h.tasks
            cfg.TABU_MAX_TL_ADD_SIZE = int(len(tasks)/8)
            cfg.TABU_MAX_TL_DROP_SIZE = int(len(tasks)/8)
            cfg.TABU_NUM_REROUTE_NEIGHBORS =  int(len(tasks)/7)
            cfg.TABU_NUM_REVERSE_NEIGHBORS = 0 # int(len(tasks)/10)
            cfg.TABU_NUM_REDISTRIBUTE_NEIGHBORS = 0 #int(len(tasks)/20)
            # cfg.TABU_MAX_TL_ADD_SIZE = 12
            # cfg.TABU_MAX_TL_DROP_SIZE = 12
            # cfg.TABU_NUM_REROUTE_NEIGHBORS = 20
            # cfg.TABU_NUM_REVERSE_NEIGHBORS = 10
            cfg.TABU_MAX_ITER = 200
            cfg.TABU_FIND_FROM_ELITE_AFTER = 200 #iterations
            cfg.TABU_RESTART_AFTER = 400 #iterations
            cfg.TABU_REVERSE_ON_CRITICAL = True
            cfg.TABU_REROUTE_ON_CRITICAL = True
            cfg.TABU_REROUTE_LONGEST_EDGE = True
            cfg.TABU_REDISTRIBUTE_PROB = 0.3
            cfg.TABU_SAVE_FITNESSES = True
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)

            data_name = data_path.split('/')[2].split('.')[0]
            init_solution_path = f'solve_results/init_data/{data_name}' #For fetch initial solution
            tik1 = time.time()
            tabu = heuristic.TabuSearch(
                processed_h, 
                c_travel, 
                c_operate, 
                heuristic.SolutionFetcher, 
                init_solution_path = init_solution_path)
            
            best_sol, best_sol_obj = tabu.search(num_robot,cfg.TABU_MAX_ITER)
            tok1 = time.time()
            tabu_time = tok1 - tik1
            print(f'Tabu time: {tabu_time}')
            if cfg.TABU_SAVE_FITNESSES:
                data_name = data_path.split('/')[2].split('.')[0]
                #add current time to data_name
                data_name += '_'+ datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S") + 'reroute_only'
                tabu.save_fitness_progress(data_name)
        
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(best_sol_obj)
            solve_log.append(np.round(tabu_time,2))
            now = datetime.datetime.now()
            solve_log.append(now)
            with open('solve_log_tabu.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

18-Sep-24 11:02:01:INFO:Solving data lim/200/LC1_2_1-5-uniform.csv with 10 robots and 106 tasks
18-Sep-24 11:02:01:INFO:TABU SEARCH PARAMETERS
18-Sep-24 11:02:01:INFO:RANDOM_SEED: 16
18-Sep-24 11:02:01:INFO:NUM_REVERSE_NEIGHBORS: 0
18-Sep-24 11:02:01:INFO:NUM_REROUTE_NEIGHBORS: 15
18-Sep-24 11:02:01:INFO:NUM_REDISTRIBUTE_NEIGHBORS: 0
18-Sep-24 11:02:01:INFO:MAX_TL_ADD_SIZE: 13
18-Sep-24 11:02:01:INFO:MAX_TL_DROP_SIZE: 13
18-Sep-24 11:02:01:INFO:MAX_TL_REDISTRIBUTE_SIZE: 10
18-Sep-24 11:02:01:INFO:MAX_ITER: 200
18-Sep-24 11:02:01:INFO:FIND_FROM_ELITE_AFTER: 200
18-Sep-24 11:02:01:INFO:RESTART_AFTER: 400
18-Sep-24 11:02:01:INFO:REROUTE_ON_CRITICAL: True
18-Sep-24 11:02:01:INFO:REVERSE_ON_CRITICAL: True
18-Sep-24 11:02:01:INFO:REVERSE_CRITICAL_LONGEST_EDGE: True
18-Sep-24 11:02:01:INFO:Initial makespan: 937.2506795011043
18-Sep-24 11:02:01:INFO:Initial fitness: 937.2506795011043
18-Sep-24 11:02:02:INFO:Best fitness updated: 927.1984664989409, chosen neighborhood: 0
18-Sep-24 11:02:02:INFO

KeyboardInterrupt: 

In [6]:
cProfile.run(tabu.search(num_robot,cfg.TABU_MAX_ITER))

18-Sep-24 11:02:22:INFO:TABU SEARCH PARAMETERS
18-Sep-24 11:02:22:INFO:RANDOM_SEED: 16
18-Sep-24 11:02:22:INFO:NUM_REVERSE_NEIGHBORS: 0
18-Sep-24 11:02:22:INFO:NUM_REROUTE_NEIGHBORS: 15
18-Sep-24 11:02:22:INFO:NUM_REDISTRIBUTE_NEIGHBORS: 0
18-Sep-24 11:02:22:INFO:MAX_TL_ADD_SIZE: 13
18-Sep-24 11:02:22:INFO:MAX_TL_DROP_SIZE: 13
18-Sep-24 11:02:22:INFO:MAX_TL_REDISTRIBUTE_SIZE: 10
18-Sep-24 11:02:22:INFO:MAX_ITER: 200
18-Sep-24 11:02:22:INFO:FIND_FROM_ELITE_AFTER: 200
18-Sep-24 11:02:22:INFO:RESTART_AFTER: 400
18-Sep-24 11:02:22:INFO:REROUTE_ON_CRITICAL: True
18-Sep-24 11:02:22:INFO:REVERSE_ON_CRITICAL: True
18-Sep-24 11:02:22:INFO:REVERSE_CRITICAL_LONGEST_EDGE: True
18-Sep-24 11:02:22:INFO:Initial makespan: 937.2506795011043
18-Sep-24 11:02:22:INFO:Initial fitness: 937.2506795011043
18-Sep-24 11:02:23:INFO:Best fitness updated: 927.1984664989409, chosen neighborhood: 0
18-Sep-24 11:02:23:INFO:Iteration 0 done
18-Sep-24 11:02:23:INFO:Best fitness updated: 920.1833172439868, chosen neighb

         2 function calls in 0.000 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.000    0.000 {built-in method builtins.exec}
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}




TypeError: exec() arg 1 must be a string, bytes or code object

In [ ]:
## mass solve using heuristics
for data_path in data_paths:
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            tik1 = time.time()
            solve_result_h = heuristic.solve_greedy_1(processed_h, num_robot)
            tok1 = time.time()
            greedy1_time = tok1 - tik1
            print(f'Greedy 1 time: {greedy1_time}')
            tik2 = time.time()
            solve_result_h_jp = heuristic.solve_greedy_jp(processed_h, num_robot)
            tok2 = time.time()
            greedy_jp_time = tok2 -tik2
            print(f'Greedy jp time: {greedy_jp_time}')
            tik3 = time.time()
            solve_result_h_jp_2 = heuristic.solve_greedy_jp_compare_start_time(processed_h, num_robot)
            tok3 = time.time()
            greedy_jp_time_2 = tok3 -tik3
            print(f'Greedy jp time 2: {greedy_jp_time_2}')

        
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(solve_result_h.get_makespan().round(2))
            solve_log.append(np.round(greedy1_time,2))
            solve_log.append(solve_result_h_jp.get_makespan().round(2))
            solve_log.append(np.round(greedy_jp_time,2))
            solve_log.append(solve_result_h_jp_2.get_makespan().round(2))
            solve_log.append(np.round(greedy_jp_time_2,2))
            with open('solve_log_h_2.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

In [ ]:
import helpers
## Mass solve 2 opt
for data_path in data_paths:
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_h = heuristic.process_greedy_1(raw)

            tik2 = time.time()
            solve_result_h2 = heuristic.solve_greedy_jp(processed_h, num_robot)
            tok2 = time.time()
            greedy_h2 = tok2 -tik2

            tik3 = time.time()
            solve_result_h3 = heuristic.solve_greedy_jp_compare_start_time(processed_h, num_robot)
            tok3 = time.time()
            greedy_jp_time_2 = tok3 -tik3

            depot = processed_h.depot
            tasks = processed_h.tasks
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)
            tik = time.time()
            solve_opt = heuristic.TwoOptSolver(processed_h,c_travel, c_operate)
            graph,makespan_2opt = solve_opt.solve(total_robots= num_robot)
            tok = time.time()
            two_opt_time = tok - tik

            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(solve_result_h2.get_makespan().round(2))
            solve_log.append(np.round(greedy_h2,2))
            solve_log.append(solve_result_h3.get_makespan().round(2))
            solve_log.append(np.round(greedy_jp_time_2,2))
            solve_log.append(makespan_2opt.round(2))
            solve_log.append(np.round(two_opt_time,2))
            now = datetime.datetime.now()
            solve_log.append(now.strftime("%Y-%m-%d %H:%M:%S"))
            with open('solve_log_h_2opt.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

In [ ]:
## Mass solve Viet 

import logging
logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')
consider_process_time =True
for data_path in data_paths:
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_h = heuristic.process_greedy_1(raw)
            depot = processed_h.depot
            tasks = processed_h.tasks
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)
            tik = time.time()
            solve_viet = heuristic.VietSolverOrig(processed_h, c_travel, c_operate)
            solve_viet.solve(num_robot,consider_process_time)
            tok = time.time()
            time_viet = tok - tik
            print(f'makespan: {solve_viet.get_makespan()}')
            print('Done')

            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(solve_viet.get_makespan().round(2))
            solve_log.append(np.round(time_viet,2))
            with open('solve_log_h_Viet_09-04-process-time.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)


In [ ]:
## Mass solve Viet k tasks

import logging
logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')

tasks_each_iters = [5]
consider_process_time =False
for task_each_ite in tasks_each_iters:
    for data_path in data_paths:
        for num_robot in num_robots:
            try:
                print(f'Solving for {data_path} with {num_robot} robots')
                raw = get_data(data_path, file_type, is_depot_task)
                processed_h = heuristic.process_greedy_1(raw)
                depot = processed_h.depot
                tasks = processed_h.tasks
                c_travel = helpers.create_task_dist_dict(depot,tasks)
                c_operate = helpers.create_operate_dict(depot,tasks)
                tik = time.time()
                solve_viet = heuristic.VGeneralizedSolver(processed_h, c_travel, c_operate)
                sol = solve_viet.solve(num_robot,task_each_ite,consider_process_time)
                tok = time.time()
                time_viet = tok - tik
                makespan = sol.get_makespan()
                print(f'makespan: {makespan}')
                print('Done')

                solve_log = []
                solve_log.append(data_path)
                solve_log.append(num_robot)
                solve_log.append(task_each_ite)
                solve_log.append(makespan.round(2))
                solve_log.append(np.round(time_viet,2))
                with open('solve_log_h_Viet k task large instances 23-04.csv', 'a') as f:
                    f.write(','.join([str(i) for i in solve_log])+'\n')
            except Exception as e:
                print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
                print(e)


In [ ]:
a ={0: 0.0, 1: 599.9817763899168, 2: 271.90904527065663, 3: 283.1551518161024, 4: 142.61641735789044, 5: 239.53321272842314, 6: 503.8917411508151, 7: 320.3896980241406, 8: 347.00099826945745, 9: 177.80785134520917, 10: 194.39276118209756, -1: 0}

In [ ]:
for i in a:
    print(i)

In [ ]:
## Mass solve Viet k tasks with random k

import logging
logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')

ite = 100
tasks_each_iter_ranges = [[1,3],[1,4],[1,5]]
consider_process_time =False
for task_each_ite in tasks_each_iter_ranges:
    for data_path in data_paths:
        for num_robot in num_robots:
            try:
                print(f'Solving for {data_path} with {num_robot} robots')
                raw = get_data(data_path, file_type, is_depot_task)
                processed_h = heuristic.process_greedy_1(raw)
                depot = processed_h.depot
                tasks = processed_h.tasks
                c_travel = helpers.create_task_dist_dict(depot,tasks)
                c_operate = helpers.create_operate_dict(depot,tasks)

                solve_result_h_jp = heuristic.solve_greedy_jp_compare_start_time(processed_h, num_robot)
                TT_makespan = solve_result_h_jp.get_makespan()

                tik = time.time()
                cur_best = np.infty
                cur_worst = -np.infty
                makespan_results = []
                no_better_TT = 0

                for i in range(ite):
                    solve_viet = heuristic.VGeneralizedSolver(processed_h, c_travel, c_operate)
                    sol = solve_viet.solve(num_robot,'random',consider_process_time,task_range=task_each_ite)
                    makespan = sol.get_makespan()
                    makespan_results.append(makespan)
                    if makespan < cur_best:
                        cur_best = makespan
                    if makespan > cur_worst:
                        cur_worst = makespan
                    if makespan <= TT_makespan:
                        no_better_TT +=1
                avg_makespan = np.mean(makespan_results)
                tok = time.time()
                time_viet = tok - tik
            
                solve_log = []
                solve_log.append(data_path)
                solve_log.append(num_robot)
                solve_log.append(str(f'{task_each_ite[0]}-{task_each_ite[1]}'))
                solve_log.append(cur_best.round(2))
                solve_log.append(cur_worst.round(2))
                solve_log.append(avg_makespan.round(2))
                solve_log.append(np.round(time_viet,2))
                solve_log.append(no_better_TT)
                with open('solve_log_h_Viet k task random large instances 29-04.csv', 'a') as f:
                    f.write(','.join([str(i) for i in solve_log])+'\n')
            except Exception as e:
                print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
                print(e)


In [ ]:
raw = get_data(data_path, file_type, is_depot_task)
processed_h = heuristic.process_greedy_1(raw)
depot = processed_h.depot
tasks = processed_h.tasks
c_travel = helpers.create_task_dist_dict(depot,tasks)
c_operate = helpers.create_operate_dict(depot,tasks)
tik = time.time()
solve_opt = heuristic.TwoOptSolver(processed_h,c_travel, c_operate)

In [ ]:
#Get data

#Read file solve_log.csv, save it as a dataframe
import pandas as pd
solve_cplex_df = pd.read_csv('solve_log.csv', header=None)
solve_cplex_df.columns = ['data_path', 'num_robot', 'solve_time', 'objective_value', 'processed_nodes', 'gap', 'solve_date']

solve_h_df = pd.read_csv('solve_log_h.csv', header=None)
solve_h_df.columns = ['data_path', 'objective_value_1', 'time_greedy_1', 'objective_value_jp', 'time_greedy_jp']

In [ ]:
# right join 2 dataframes on column data_path
merged_df = pd.merge(solve_cplex_df, solve_h_df, on='data_path', how='right')

In [ ]:
merged_df['diff_h_1'] = np.round(1- merged_df['objective_value_1']/merged_df['objective_value'],2)
merged_df['diff_h_jp'] = np.round(1- merged_df['objective_value_jp']/merged_df['objective_value'],2)
#change order of columns in meged_df as follow: data_path, num_robot, objective_value, solve_time, objective_value_1, time_greedy_1, diff_h_1, objective_value_jp, time_greedy_jp, diff_h_jp,
merged_df = merged_df[['data_path', 'num_robot', 'objective_value', 'solve_time', 'objective_value_1', 'time_greedy_1', 'diff_h_1', 'objective_value_jp', 'time_greedy_jp', 'diff_h_jp']]
merged_df
#Remove the 'lim' prefix from data_path
merged_df['data_path'] = merged_df['data_path'].str.replace('lim/','')
#round solve_time to 2 decimal places
merged_df['solve_time'] = np.round(merged_df['solve_time'],2)
#round objective_value to 2 decimal places
merged_df['objective_value'] = np.round(merged_df['objective_value'],2)

In [ ]:
merged_df.to_latex('merged_df.tex', index=True)

In [ ]:
#Temp
import pandas as pd
solve_h_df = pd.read_csv('solve_log_h_2.csv', header=None)
solve_h_df.columns = ['data_path', 'objective_value_1', 'time_greedy_1', 'objective_value_2', 'time_greedy_2', 'objective_value_3', 'time_greedy_3']
#Keep the following columns: data_path, objective_value_2, time_greedy_2
solve_h_df = solve_h_df[['data_path', 'objective_value_2', 'time_greedy_2']]

In [ ]:
solve_h_df_2opt = pd.read_csv('solve_log_h_2opt.csv', header=None)
solve_h_df_2opt.columns = ['data_path', 'num_robot', 'objective_value_3', 'time_greedy_3', 'objective_value_2opt', 'time_2opt']

In [ ]:
#Join 2 dataframes on data_path
merged_df_2 = pd.merge(solve_h_df, solve_h_df_2opt, on='data_path', how='right')
#Change the order of columns as follow: data_path, num_robot, objective_value_2, time_greedy_2, objective_value_3, time_greedy_3, objective_value_2opt, time_2opt
merged_df_2 = merged_df_2[['data_path', 'num_robot', 'objective_value_2', 'time_greedy_2', 'objective_value_3', 'time_greedy_3', 'objective_value_2opt', 'time_2opt']]

In [ ]:
#save the data as a latex table
merged_df_2.to_latex('merged_df_2.tex', index=True)